# Velocity-weakening Workbench

Use this notebook to inspect an existing run or launch a new one without typing a long CLI command. The notebook calls the same `velocity_weakening_tatva.py` driver you have been using in the terminal, but exposes the most common settings as Python variables.


## 1. Imports and Helper Functions

Run this cell first. It loads the helper utilities from `src/velocity_weakening_notebook_helpers.py`.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

from velocity_weakening_notebook_helpers import (
    DEFAULT_MU_DISP_Y_POINTS,
    REPO_ROOT,
    RUNS_ROOT,
    build_driver_command,
    default_case_config,
    launch_case,
    list_runs,
    load_summary_for_run,
    print_runs,
    resolve_run,
    run_paths,
)

print(f"Repo root: {REPO_ROOT}")
print(f"Runs root: {RUNS_ROOT}")
print(f"Default mu-disp y-points: {DEFAULT_MU_DISP_Y_POINTS}")


Repo root: /Volumes/Gauss-T7/tatva
Runs root: /Volumes/Gauss-T7/tatva/Velocity-weakening/runs
Default mu-disp y-points: [125.0, 250.0, 375.0, 450.0]


## 2. Inspect an Existing Run

You can locate a run in three ways:

- `RUN_NUMBER`: for example `5`
- `RUN_SUFFIX`: the part after `0005_...`
- `RUN_NAME`: the full folder name

Usually `RUN_NUMBER` is enough.


In [2]:
RUN_NUMBER = 5
RUN_SUFFIX = "lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shearx2p5-mesh2mm"
RUN_NAME = None
LIST_LIMIT = 10

In [3]:
print_runs(limit=LIST_LIMIT)

run_dir = resolve_run(
    run_number=RUN_NUMBER,
    run_suffix=RUN_SUFFIX,
    run_name=RUN_NAME,
)
summary_payload = load_summary_for_run(run_dir)
paths = run_paths(run_dir)

print(f"\nResolved run: {run_dir.name}")
print(json.dumps(paths, indent=2))

summary = summary_payload["summary"]
important = {
    "mesh_size": summary.get("mesh_size"),
    "dt": summary.get("dt"),
    "saved_frames": summary.get("saved_frames"),
    "pressure_frames_saved": summary.get("pressure_frames_saved"),
    "shear_frames_saved": summary.get("shear_frames_saved"),
    "final_max_slip": summary.get("final_max_slip"),
    "final_avg_tau": summary.get("final_avg_tau"),
    "final_avg_sigma_n": summary.get("final_avg_sigma_n"),
}
important


run_id  mesh_size  saved_frames  dt                     status     run_name                                                                          
-----------------------------------------------------------------------------------------------------------------------------------------------------
42      5.0        100855        6.246464498000369e-08  completed  0042_lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm
41      5.0        100855        6.246464498000369e-08  completed  0041_lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm
40      5.0        120000        6.246464498000369e-08  completed  0040_lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm
39      5.0        120000        6.246464498000369e-08  completed  0039_lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm
38      5.0        120000        6.246464498000369e-08  completed  0038_lockedge-normal40ms-ramp20ms

{'mesh_size': 2.0,
 'dt': 1.7667669107319865e-08,
 'saved_frames': 19825,
 'pressure_frames_saved': None,
 'shear_frames_saved': None,
 'final_max_slip': 17.701011657714844,
 'final_avg_tau': 0.4486418664455414,
 'final_avg_sigma_n': 0.7477364540100098}

## 3. Prepare a New Case

Edit only the values you care about. Everything else falls back to the current default workflow:

- CPU execution through `conda run -n tatva`
- `2D` or `3D` selection with configurable thickness
- direct control of `normal stress`, `shear_tau_k`, and `shear_tau_s`
- PDF plots
- 4K `von Mises` and `sigma_xy` animations
- four-panel `mu-disp` plot at `y = 125, 250, 375, 450 mm`


In [4]:
NEW_CASE = default_case_config()

ms = 1e-3

# Common fields to edit
NEW_CASE["run_label"] = "lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm"
NEW_CASE["dimension"] = 2
NEW_CASE["thickness"] = 0.0
NEW_CASE["mesh_size"] = 5.0
NEW_CASE["normal_stress"] = 16.0
NEW_CASE["shear_tau_k"] = 12
NEW_CASE["shear_tau_s"] = 24
NEW_CASE["normal_phase_time"] = 0.04
NEW_CASE["normal_ramp_time"] = 0.02
NEW_CASE["tau_k_start_fraction"] = 0.75
NEW_CASE["tau_k_full_fraction"] = 1.00
NEW_CASE["shear_phase_time"] = 20 * ms
NEW_CASE["shear_ramp_time"] = NEW_CASE["shear_phase_time"] - 0.5 * ms
NEW_CASE["frames_per_phase"] = 4800
NEW_CASE["shear_frames_per_phase"] = 28800 * 16
NEW_CASE["num_threads"] = 16
NEW_CASE["animation_workers"] = 8

# Advanced fields (uncomment and edit if needed)
# NEW_CASE["dimension"] = 3
# NEW_CASE["thickness"] = 50.0
# NEW_CASE["normal_stress"] = 16.0
# NEW_CASE["shear_tau_k"] = 82.75862068965516
# NEW_CASE["shear_tau_s"] = 110.34482758620689
# NEW_CASE["normal_penalty"] = 38310.0
# NEW_CASE["tangential_penalty"] = 3831.0
NEW_CASE["skip_animation"] = True
NEW_CASE["skip_shear_stress_animation"] = True
# NEW_CASE["mu_disp_y_points"] = [125.0, 250.0, 375.0, 450.0]

NEW_CASE

{'dimension': 2,
 'thickness': 0.0,
 'mesh_size': 5.0,
 'cfl': 0.35,
 'dtype': 'float32',
 'num_threads': 16,
 'normal_stress': 16.0,
 'shear_tau_k': 12,
 'shear_tau_s': 24,
 'normal_phase_time': 0.04,
 'normal_ramp_time': 0.02,
 'shear_phase_time': 0.02,
 'tau_k_start_fraction': 0.75,
 'tau_k_full_fraction': 1.0,
 'shear_ramp_time': 0.0195,
 'shear_scale': 2.5,
 'lock_shear_edge_during_normal': True,
 'frames_per_phase': 4800,
 'shear_frames_per_phase': 460800,
 'omit_initial_frame': True,
 'animation_fps': 60,
 'animation_workers': 8,
 'animation_dpi': 320,
 'animation_width': 12.0,
 'animation_height': 6.75,
 'stress_percentile': 99.5,
 'animation_margin': 8.0,
 'output_prefix': 'simulation',
 'run_label': 'lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm',
 'mu_disp_y_points': [125.0, 250.0, 375.0, 450.0],
 'skip_mu_plot': False,
 'skip_mu_disp_plot': False,
 'skip_animation': True,
 'skip_shear_stress_animation': True,
 'animation_swap_axes': True}

In [5]:
cmd = build_driver_command(NEW_CASE)
print("Command preview:\n")
print(" ".join(cmd))

Command preview:

conda run -n tatva python /Volumes/Gauss-T7/tatva/Velocity-weakening/src/velocity_weakening_tatva.py --platform cpu --dimension 2 --thickness 0.0 --mesh-size 5.0 --cfl 0.35 --dtype float32 --num-threads 16 --normal-stress 16.0 --shear-tau-k 12 --shear-tau-s 24 --normal-phase-time 0.04 --normal-ramp-time 0.02 --tau-k-start-fraction 0.75 --tau-k-full-fraction 1.0 --shear-phase-time 0.02 --shear-scale 2.5 --frames-per-phase 4800 --shear-frames-per-phase 460800 --animation-fps 60 --animation-workers 8 --animation-dpi 320 --animation-width 12.0 --animation-height 6.75 --stress-percentile 99.5 --animation-margin 8.0 --output-prefix simulation --run-label lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm --shear-ramp-time 0.0195 --mu-disp-y-points 125.0 250.0 375.0 450.0 --lock-shear-edge-during-normal --omit-initial-frame --skip-animation --skip-shear-stress-animation


## 4. Launch the New Case

Set `RUN_NEW_CASE = True` only when you are ready. This will run the full simulation and post-processing pipeline.


In [ ]:
RUN_NEW_CASE = True

if RUN_NEW_CASE:
    launch_case(NEW_CASE, stream_output=True)
else:
    print("Set RUN_NEW_CASE = True when you want to launch the case.")

Launching:
conda run -n tatva python /Volumes/Gauss-T7/tatva/Velocity-weakening/src/velocity_weakening_tatva.py --platform cpu --dimension 2 --thickness 0.0 --mesh-size 5.0 --cfl 0.35 --dtype float32 --num-threads 16 --normal-stress 16.0 --shear-tau-k 12 --shear-tau-s 24 --normal-phase-time 0.04 --normal-ramp-time 0.02 --tau-k-start-fraction 0.75 --tau-k-full-fraction 1.0 --shear-phase-time 0.02 --shear-scale 2.5 --frames-per-phase 4800 --shear-frames-per-phase 460800 --animation-fps 60 --animation-workers 8 --animation-dpi 320 --animation-width 12.0 --animation-height 6.75 --stress-percentile 99.5 --animation-margin 8.0 --output-prefix simulation --run-label lockedge-normal40ms-ramp20ms-hold20ms-shear0p530994ms-shear20-shear40-mesh5mm --shear-ramp-time 0.0195 --mu-disp-y-points 125.0 250.0 375.0 450.0 --lock-shear-edge-during-normal --omit-initial-frame --skip-animation --skip-shear-stress-animation
